# Notebook 01b — Data Acquisition (Mendeley Fallback)

**Project:** Explainable Exoplanet Transit Classification  
**Output:** `kepler_gaf_dataset.npz` — train/val/test splits of 64×64 GAF images ready for ViT-B/16

**Why this notebook exists:**  
Notebook `01_data_acquisition.ipynb` downloads light curves live from MAST via Lightkurve.  
That takes 15+ hours across multiple Kaggle sessions and the cache is wiped on each restart.  
This notebook uses the preprocessed dataset from **Macedo & Zalewski (2024)** instead — runs in under 5 minutes.

**Dataset:** Mendeley DOI `10.17632/wctcv34962.3`  
**File used:** `all_global.csv` — 5,302 phase-folded Kepler light curves (2001 bins each), labels included.

**Setup before running on Kaggle:**  
1. Upload the `Dataset_Machine_Learning_Exoplanets_2024` folder as a Kaggle dataset  
2. Add it to this notebook via sidebar → **Add Data → Your Datasets**  
3. Update `MENDELEY_INPUT` in Section 2 to match your dataset name  

### Pipeline
```
Mendeley all_global.csv  →  5,302 phase-folded light curves (2001 bins, labelled)
                         →  filter to CONFIRMED / FALSE POSITIVE
                         →  rescale each row to [-1, 1]
pyts                     →  Gramian Angular Field (2001-bin → 64×64 image)
scikit-learn             →  stratified 70 / 15 / 15 split
numpy .npz               →  saved dataset
```

## Section 1 — Install & Imports

In [ ]:
# lightkurve is NOT needed for this notebook
!pip install pyts tqdm -q

In [ ]:
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from tqdm.auto import tqdm

from pyts.image import GramianAngularField
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
print("All imports OK")

## Section 2 — Configuration

In [ ]:
# ── UPDATE THIS PATH to match your Kaggle dataset name ──────────────────────
MENDELEY_INPUT = Path('/kaggle/input/exoplanets/Dataset_Machine_Learning_Exoplanets_2024')
# ─────────────────────────────────────────────────────────────────────────────

CFG = {
    'phase_bins'  : 2001,  # all_global.csv has 2001 flux columns per row
    'gaf_size'    : 64,    # output GAF image size (64×64)
    'test_size'   : 0.15,
    'val_size'    : 0.15,
    'random_seed' : 42,
    'data_dir'    : Path('/kaggle/working/data'),
}

CFG['data_dir'].mkdir(parents=True, exist_ok=True)

print("Config:")
for k, v in CFG.items():
    print(f"  {k}: {v}")

print(f"\nMendeley path: {MENDELEY_INPUT}")
print(f"Path exists:   {MENDELEY_INPUT.exists()}")

## Section 3 — Load Mendeley Data

`all_global.csv` contains 5,302 phase-folded Kepler light curves.  
- Columns `0` to `2000`: flux values at 2001 evenly-spaced phase bins  
- Column `label`: `CONFIRMED` or `FALSE POSITIVE` (no candidates — already filtered by Macedo & Zalewski)  
- No missing values  

This section produces:
- `sequences` — list of float32 arrays, shape `(2001,)`, rescaled to `[-1, 1]`
- `valid_df` — DataFrame with `label` (int) and `koi_disposition` (string) columns

In [ ]:
GLOBAL_PATH = MENDELEY_INPUT / 'all_global.csv'

print(f"Loading {GLOBAL_PATH} ...")
raw_df = pd.read_csv(GLOBAL_PATH)

print(f"Shape: {raw_df.shape}")
print(f"Label distribution:")
print(raw_df['label'].value_counts())
print(f"NaN count: {raw_df.isna().sum().sum()}")

In [ ]:
# Extract flux columns (named '0' to '2000') and rescale each row to [-1, 1]
flux_cols = [str(i) for i in range(CFG['phase_bins'])]

flux_matrix = raw_df[flux_cols].values.astype(np.float32)  # (5302, 2001)

# Per-row min-max rescale to [-1, 1]
f_min = flux_matrix.min(axis=1, keepdims=True)
f_max = flux_matrix.max(axis=1, keepdims=True)
denom = f_max - f_min

# Drop flat-line rows (no signal variation)
flat_mask = denom.squeeze() >= 1e-8
flux_matrix = flux_matrix[flat_mask]
f_min = f_min[flat_mask]
f_max = f_max[flat_mask]
denom = denom[flat_mask]
labels_raw = raw_df['label'].values[flat_mask]

flux_scaled = 2.0 * (flux_matrix - f_min) / denom - 1.0

print(f"Rows after flat-line filter: {flat_mask.sum()} / {len(raw_df)}")
print(f"Flux range after rescaling: [{flux_scaled.min():.3f}, {flux_scaled.max():.3f}]")

# Verify range
assert flux_scaled.min() >= -1.0 - 1e-5 and flux_scaled.max() <= 1.0 + 1e-5, \
    "Rescaling failed — values outside [-1, 1]"
print("Range check passed.")

In [ ]:
# Build sequences list and valid_df
sequences = [flux_scaled[i] for i in range(len(flux_scaled))]

valid_df = pd.DataFrame({
    'koi_disposition': labels_raw,
    'label': (labels_raw == 'CONFIRMED').astype(int),
})

print(f"Sequences: {len(sequences)}  shape of each: {sequences[0].shape}")
print(f"Class balance: {valid_df['label'].mean():.1%} confirmed")
print(valid_df['koi_disposition'].value_counts())

assert len(sequences) == len(valid_df)
assert sequences[0].shape == (CFG['phase_bins'],)
assert sequences[0].dtype == np.float32
print("\nSection 3 checks passed.")

In [ ]:
# Plot a few phase-folded light curves to verify they look sensible
fig, axes = plt.subplots(2, 3, figsize=(14, 5))
for cls, cls_name, row_axes in zip([1, 0], ['CONFIRMED', 'FALSE POSITIVE'], [axes[0], axes[1]]):
    indices = np.where(valid_df['label'].values == cls)[0][:3]
    for ax, idx in zip(row_axes, indices):
        ax.plot(sequences[idx], lw=0.7)
        ax.set_title(f'{cls_name}', fontsize=9)
        ax.set_xlabel('Phase bin')
        ax.set_ylabel('Flux')
fig.suptitle('Phase-folded light curves (top: CONFIRMED, bottom: FALSE POSITIVE)', fontsize=11)
plt.tight_layout()
plt.savefig(CFG['data_dir'] / 'sample_lightcurves.png', dpi=120)
plt.show()

## Section 4 — GAF Image Generation

Converts each 1D phase-folded sequence to a 2D Gramian Angular Field image.  
The GAF encodes temporal correlations as a matrix of angular cosine products, preserving the transit dip shape.

Reference: Choudhary et al. 2025 validated this exact transformation on Kepler data.

In [ ]:
print(f"Converting {len(sequences)} sequences → {CFG['gaf_size']}×{CFG['gaf_size']} GAF images...")

gaf_transform = GramianAngularField(image_size=CFG['gaf_size'], method='summation')

X_raw = np.stack(sequences)                                          # (N, 2001)
X     = gaf_transform.fit_transform(X_raw).astype(np.float32)       # (N, 64, 64)
y     = valid_df['label'].values.astype(np.int64)

print(f"X: {X.shape}  dtype={X.dtype}")
print(f"y: {y.shape}  confirmed={y.mean():.1%}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for cls, cls_name, row_axes in zip([1, 0], ['CONFIRMED', 'FALSE POSITIVE'], axes):
    indices = np.where(y == cls)[0][:5]
    for ax, idx in zip(row_axes, indices):
        ax.imshow(X[idx], cmap='viridis', origin='lower', vmin=-1, vmax=1)
        ax.set_title(cls_name, fontsize=8)
        ax.axis('off')
fig.suptitle('Sample GAF Images (top: CONFIRMED, bottom: FALSE POSITIVE)', fontsize=11)
plt.tight_layout()
plt.savefig(CFG['data_dir'] / 'sample_gaf_images.png', dpi=120)
plt.show()

## Section 5 — Stratified Train / Val / Test Split

70 % train · 15 % val · 15 % test, stratified on label.

In [ ]:
X_tv, X_test, y_tv, y_test = train_test_split(
    X, y,
    test_size=CFG['test_size'],
    stratify=y,
    random_state=CFG['random_seed'],
)

val_fraction = CFG['val_size'] / (1.0 - CFG['test_size'])
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv,
    test_size=val_fraction,
    stratify=y_tv,
    random_state=CFG['random_seed'],
)

for name, X_s, y_s in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    print(f"{name:5s}: {len(X_s):4d} samples  |  confirmed: {y_s.mean():.1%}")

## Section 6 — Save Dataset

In [ ]:
DATASET_PATH  = CFG['data_dir'] / 'kepler_gaf_dataset.npz'
METADATA_PATH = CFG['data_dir'] / 'valid_kois.csv'

np.savez_compressed(
    DATASET_PATH,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test,
)
valid_df.to_csv(METADATA_PATH, index=False)

size_mb = DATASET_PATH.stat().st_size / (1024 ** 2)
print(f"Saved dataset:  {DATASET_PATH}  ({size_mb:.1f} MB)")
print(f"Saved metadata: {METADATA_PATH}")

## Section 7 — Verification

In [ ]:
data = np.load(DATASET_PATH)

print("Dataset verification")
print("-" * 45)
for split in ['train', 'val', 'test']:
    X_s = data[f'X_{split}']
    y_s = data[f'y_{split}']
    print(
        f"{split:5s}  X={str(X_s.shape):16s}  "
        f"dtype={X_s.dtype}  "
        f"confirmed={y_s.mean():.1%}  "
        f"range=[{X_s.min():.2f}, {X_s.max():.2f}]"
    )

print()
print("Ready for Notebook 02 — Model Evaluation")